<a href="https://colab.research.google.com/github/qefdva/EldwhrPtksrl/blob/main/audwhrPtksrl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install gradio -q

In [6]:
# 필요한 라이브러리 설치 (Colab 환경에서 최초 1회 실행 필요)
!pip install gradio -q

import gradio as gr
from itertools import combinations

# --- 1. 상품 데이터 정의 ---

one_time_packages = {
    "반역자의 새겨진 그림자 컬렉션 Ⅰ": {"price": 5900, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 5, "별의소리": 0},
    "반역자의 꿈을 잡는 컬렉션": {"price": 19000, "금빛": 0, "울린": 0, "꿈": 10, "새겨진": 0, "별의소리": 400},
    "반역자의 새겨진 그림자 컬렉션 Ⅱ": {"price": 19000, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 10, "별의소리": 400},
    "TARD-E의 월간 지원": {"price": 25000, "금빛": 5, "울린": 5, "꿈": 0, "새겨진": 0, "별의소리": 500},
    "탐구의 금빛 파도 컬렉션 Ⅰ": {"price": 12000, "금빛": 5, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 400},
    "탐구의 조수 컬렉션": {"price": 12000, "금빛": 0, "울린": 5, "꿈": 0, "새겨진": 0, "별의소리": 0},
    "탐구의 금빛 파도 커렉션 Ⅱ": {"price": 37000, "금빛": 15, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 500},
    "달빛x60(초회)": {"price": 1200, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 120},
    "달빛x300(초회)": {"price": 5900, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 600},
    "달빛x980(초회)": {"price": 19000, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 1960},
    "달빛x1980(초회)": {"price": 37000, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 3960},
    "달빛x3280(초회)": {"price": 65000, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 6560},
    "달빛x6480(초회)": {"price": 119000, "금빛": 0, "울린": 0, "꿈": 0, "새겨진": 0, "별의소리": 12960}
}

repeat_packages = {
    "달빛x60": {"price": 1200, "별의소리": 60},
    "달빛x300": {"price": 5900, "별의소리": 330},
    "달빛x980": {"price": 19000, "별의소리": 1090},
    "달빛x1980": {"price": 37000, "별의소리": 2240},
    "달빛x3280": {"price": 65000, "별의소리": 3880},
    "달빛x6480": {"price": 119000, "별의소리": 8080}
}

# --- 2. 동적 계획법(DP) 사전 계산 ---
MAX_ASTRITE = 150000
dp = [float('inf')] * (MAX_ASTRITE + 8081)
dp[0] = 0
items_used = [None] * (MAX_ASTRITE + 8081)
prev_idx = [0] * (MAX_ASTRITE + 8081)

for i in range(MAX_ASTRITE + 1):
    if dp[i] == float('inf'): continue
    for name, data in repeat_packages.items():
        nxt = i + data['별의소리']
        if dp[i] + data['price'] < dp[nxt]:
            dp[nxt] = dp[i] + data['price']
            items_used[nxt] = name
            prev_idx[nxt] = i

# --- 3. 핵심 계산 알고리즘 ---
def calculate_optimal_purchase(req_star, req_gold, req_echo, req_dream, req_shadow, bought_items):
    req_star = int(req_star) if req_star else 0
    req_gold = int(req_gold) if req_gold else 0
    req_echo = int(req_echo) if req_echo else 0
    req_dream = int(req_dream) if req_dream else 0
    req_shadow = int(req_shadow) if req_shadow else 0

    total_required_astrite_value = req_star + (req_gold + req_echo + req_dream + req_shadow) * 160

    if total_required_astrite_value == 0:
        return "구매 희망 재화를 입력해주세요."

    available_one_time = [k for k in one_time_packages.keys() if k not in bought_items]

    best_total_price = float('inf')
    best_total_astrite_value = -1 # 가성비(더 많은 재화) 비교용

    best_combo_names = []
    best_repeat_path = []
    best_obtained_resources = (0, 0, 0, 0, 0) # 금빛, 울린, 꿈, 새겨진, 별의소리

    # 1회성 상품 조합 탐색
    for r in range(len(available_one_time) + 1):
        for combo in combinations(available_one_time, r):
            combo_price = 0
            got_gold, got_echo, got_dream, got_shadow, got_star = 0, 0, 0, 0, 0

            for item in combo:
                p_data = one_time_packages[item]
                combo_price += p_data["price"]
                got_gold += p_data["금빛"]
                got_echo += p_data["울린"]
                got_dream += p_data["꿈"]
                got_shadow += p_data["새겨진"]
                got_star += p_data["별의소리"]

            # 무늬 부족분 계산 -> 별의 소리 요구량으로 환산
            def_gold = max(0, req_gold - got_gold)
            def_echo = max(0, req_echo - got_echo)
            def_dream = max(0, req_dream - got_dream)
            def_shadow = max(0, req_shadow - got_shadow)

            needed_astrite_for_tides = (def_gold + def_echo + def_dream + def_shadow) * 160
            total_astrite_deficit = max(0, (req_star + needed_astrite_for_tides) - got_star)

            if total_astrite_deficit > MAX_ASTRITE:
                return "요구 수량이 너무 큽니다. 범위를 줄여주세요."

            # DP를 이용해 남은 별의 소리를 채우는 최소 비용 탐색
            min_repeat_price = float('inf')
            best_idx = total_astrite_deficit

            for i in range(total_astrite_deficit, total_astrite_deficit + 8081):
                if dp[i] < min_repeat_price:
                    min_repeat_price = dp[i]
                    best_idx = i
                # 수정 1: 가격이 같다면 재화를 더 많이 주는(i가 더 큰) 방식을 채택
                elif dp[i] == min_repeat_price and i > best_idx:
                    best_idx = i

            total_price = combo_price + min_repeat_price

            # 이번 조합의 최종 획득 총 가치 (별의 소리 기준)
            current_total_astrite_value = (got_star + best_idx) + (got_gold + got_echo + got_dream + got_shadow) * 160

            # 수정 1: 가격이 더 싸거나, 가격은 같은데 재화를 더 많이 준다면 최적해 갱신
            if total_price < best_total_price or (total_price == best_total_price and current_total_astrite_value > best_total_astrite_value):
                best_total_price = total_price
                best_total_astrite_value = current_total_astrite_value
                best_combo_names = list(combo)
                best_obtained_resources = (got_gold, got_echo, got_dream, got_shadow, got_star + best_idx)

                path = []
                curr = best_idx
                while curr > 0:
                    path.append(items_used[curr])
                    curr = prev_idx[curr]
                best_repeat_path = path

    # --- 4. 결과 출력 포맷팅 (수정 3 반영) ---
    result_md = f"### 💡 최적 구매 전략 추천 결과\n"
    result_md += f"**예상 총 결제 금액: {best_total_price:,}원**\n\n"

    result_md += "#### 🛒 구매해야 할 상품 목록\n"
    if best_combo_names:
        for item in best_combo_names:
            result_md += f"- **{item}** ({one_time_packages[item]['price']:,}원)\n"

    repeat_counts = {}
    for item in best_repeat_path:
        repeat_counts[item] = repeat_counts.get(item, 0) + 1

    for item, count in sorted(repeat_counts.items(), key=lambda x: repeat_packages[x[0]]['price'], reverse=True):
         result_md += f"- **{item}** x{count}개 (총 {repeat_packages[item]['price'] * count:,}원)\n"

    if not best_combo_names and not best_repeat_path:
         result_md += "- 추가로 구매할 상품이 없습니다 (이미 조건을 만족함).\n"

    # 획득 재화 상세 표기
    obt_gold, obt_echo, obt_dream, obt_shadow, obt_star = best_obtained_resources

    result_md += "\n---\n"
    result_md += "#### 💎 획득하게 되는 실제 재화 양\n"
    if obt_star > 0: result_md += f"- **별의 소리:** {obt_star:,} 개\n"
    if obt_gold > 0: result_md += f"- **금빛 파도의 무늬:** {obt_gold:,} 개\n"
    if obt_echo > 0: result_md += f"- **울린 조수의 무늬:** {obt_echo:,} 개\n"
    if obt_dream > 0: result_md += f"- **꿈을 잡는 무늬:** {obt_dream:,} 개\n"
    if obt_shadow > 0: result_md += f"- **새겨진 그림자의 무늬:** {obt_shadow:,} 개\n"

    # 초과량 및 가성비 계산
    surplus_astrite_value = best_total_astrite_value - total_required_astrite_value
    efficiency = best_total_price / best_total_astrite_value if best_total_astrite_value > 0 else 0

    result_md += "\n#### 📊 종합 분석\n"
    result_md += f"- **모든 재화를 '별의 소리' 가치로 치환한 총량:** **{best_total_astrite_value:,} 개**\n"
    result_md += f"- **구매 효율:** 1 별의 소리당 약 **{efficiency:.2f}원**\n"

    # 수정 2 & 3: 초과량 표기 방식 변경
    if surplus_astrite_value > 0:
        result_md += f"- **초과 구매 여부:** 상품 구조상 목표치보다 **{surplus_astrite_value:,} 별의 소리(가치)만큼 초과 획득**하게 됩니다.\n"
    else:
        result_md += f"- **초과 구매 여부:** 낭비되는 초과량 없이 **정확히 딱 맞게** 구매됩니다.\n"

    return result_md

# --- 5. Gradio 웹 UI 구성 ---
with gr.Blocks(theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🌊 명조: 워더링 웨이브 재화 구매 최적화 계산기")
    gr.Markdown("목표로 하는 재화 수량과 이미 구매 완료한 패키지를 선택하면, 가장 저렴하게 구매하는 방법을 알려드립니다.")

    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🎯 구매 희망 수량 입력")
            req_star = gr.Number(label="별의 소리", value=0, precision=0)
            req_gold = gr.Number(label="금빛 파도의 무늬", value=0, precision=0)
            req_echo = gr.Number(label="울린 조수의 무늬", value=0, precision=0)
            req_dream = gr.Number(label="꿈을 잡는 무늬", value=0, precision=0)
            req_shadow = gr.Number(label="새겨진 그림자의 무늬", value=0, precision=0)

            gr.Markdown("### ☑️ 이미 구매한 1회성 상품 체크")
            bought_items = gr.CheckboxGroup(
                choices=list(one_time_packages.keys()),
                label="아래 목록 중 '이미 구매해서 더 이상 살 수 없는 상품'을 체크해주세요."
            )

            calc_btn = gr.Button("최적 구매 전략 계산하기", variant="primary")

        with gr.Column():
            result_output = gr.Markdown("결과가 여기에 표시됩니다.")

    calc_btn.click(
        fn=calculate_optimal_purchase,
        inputs=[req_star, req_gold, req_echo, req_dream, req_shadow, bought_items],
        outputs=result_output
    )

app.launch(debug=True)

/tmp/ipykernel_9395/758862434.py:177: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://82aa550ee3111285ef.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://82aa550ee3111285ef.gradio.live
